In [10]:
import torch
import numpy as np
import pandas as pd
import joblib
import os

In [11]:
import os
import random
import numpy as np

import mlflow
import mlflow.pytorch
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [12]:
from torch import nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from torch.utils.data import TensorDataset, DataLoader

In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [14]:
# Для локального стенда из docker-compose
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID", "admin")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY", "password")
raw_s3_endpoint = os.getenv("MLFLOW_S3_ENDPOINT_URL", "http://localhost:9000")

# `minio` резолвится только внутри docker-сети. Для локального ноутбука нужен localhost.
if "minio:9000" in raw_s3_endpoint:
    raw_s3_endpoint = "http://localhost:9000"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = raw_s3_endpoint

# Если на сервере включена basic-auth, задайте логин/пароль
# os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("MLFLOW_TRACKING_USERNAME", "admin")
# os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("MLFLOW_TRACKING_PASSWORD", "password")

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5050")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print("MLflow URI:", mlflow.get_tracking_uri())

MLflow URI: http://localhost:5050


In [15]:
# Для воспроизводимости
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [16]:
X_data = pd.read_csv('../formatted_data/X_final.csv')
y_data = pd.read_csv('../formatted_data/y_final.csv')

In [17]:
print(f"X shape: {X_data.shape}")
print(f"y shape: {y_data.shape}")

X shape: (339055, 78)
y shape: (339055, 13)


In [18]:
X = X_data
y = y_data
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (339055, 78)
y shape: (339055, 13)


In [19]:
RANDOM_STATE = 42
target_cols = y.columns.tolist()

mask = y[target_cols].notna().all(axis=1)
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

mask = y[target_cols].notna().all(axis=1)

X_clean = X_scaled[mask]
y_clean = y.loc[mask, target_cols].values

In [20]:
def masked_bce_loss(outputs, targets):
    mask = ~torch.isnan(targets)

    targets = torch.where(mask, targets, torch.zeros_like(targets))

    loss = nn.BCEWithLogitsLoss(reduction='none')(outputs, targets)

    loss = loss * mask
    return loss.sum() / mask.sum()

In [ ]:
def train_and_validate(
    model,
    optimizer,
    train_loader,
    val_loader,
    num_epochs,
    device,
    registered_model_name,
    experiment_name,
    verbose=True,
):

    mlflow.set_experiment(experiment_name)

    client = MlflowClient()

    with mlflow.start_run(run_name=registered_model_name):



        train_losses = []
        val_losses = []
        val_rocs = []


        for epoch in range(1, num_epochs + 1):

            # ================= TRAIN =================

            model.train()

            running_loss = 0

            for X_batch, y_batch in train_loader:

                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                optimizer.zero_grad()

                outputs = model(X_batch)

                loss = masked_bce_loss(outputs, y_batch)

                loss.backward()

                optimizer.step()

                running_loss += (
                    loss.item() * X_batch.size(0)
                )

            train_loss = (
                running_loss / len(train_loader.dataset)
            )

            train_losses.append(train_loss)

            # ================= VALID =================

            model.eval()

            running_loss = 0

            all_preds = []
            all_targets = []

            with torch.no_grad():

                for X_batch, y_batch in val_loader:

                    X_batch = X_batch.to(device)
                    y_batch = y_batch.to(device)

                    outputs = model(X_batch)

                    loss = masked_bce_loss(outputs, y_batch)

                    running_loss += (
                        loss.item() * X_batch.size(0)
                    )

                    probs = torch.sigmoid(outputs)

                    all_preds.append(
                        probs.cpu().numpy()
                    )

                    all_targets.append(
                        y_batch.cpu().numpy()
                    )

            val_loss = (
                running_loss / len(val_loader.dataset)
            )

            val_losses.append(val_loss)

            # ================= ROC-AUC =================

            all_preds = np.vstack(all_preds)
            all_targets = np.vstack(all_targets)

            roc_scores = []

            for i in range(all_targets.shape[1]):

                mask = ~np.isnan(all_targets[:, i])

                if mask.sum() == 0:
                    continue

                if len(np.unique(all_targets[mask, i])) < 2:
                    continue

                roc = roc_auc_score(
                    all_targets[mask, i],
                    all_preds[mask, i]
                )

                roc_scores.append(roc)

            mean_roc = np.mean(roc_scores)

            val_rocs.append(mean_roc)

            mlflow.log_metric(
                "train_loss",
                float(train_loss),
                step=epoch
            )

            mlflow.log_metric(
                "val_loss",
                float(val_loss),
                step=epoch
            )

            mlflow.log_metric(
                "val_roc_auc",
                float(mean_roc),
                step=epoch
            )

            if verbose:

                print(
                    f"Epoch {epoch} | "
                    f"Train Loss={train_loss:.4f} | "
                    f"Val Loss={val_loss:.4f} | "
                    f"ROC-AUC={mean_roc:.4f}"
                )


        mlflow.log_metric(
            "final_val_roc_auc",
            float(val_rocs[-1])
        )


        sample_input = next(iter(val_loader))[0][:4]

        sample_input_np = sample_input.numpy()

        model.eval()

        with torch.no_grad():

            sample_output = torch.sigmoid(
                model(sample_input.to(device))
            ).cpu().numpy()

        signature = infer_signature(
            sample_input_np,
            sample_output
        )

        model_info = mlflow.pytorch.log_model(
            pytorch_model=model,
            artifact_path="model",
            signature=signature,
            input_example=sample_input_np,
            registered_model_name=registered_model_name,
        )

        new_version = model_info.registered_model_version
        client.set_model_version_tag(registered_model_name, new_version, "env", "PRD")
        client.set_registered_model_alias(registered_model_name, "prd", new_version)

        # model_info = mlflow.pytorch.log_model(
        #     pytorch_model=model,
        #     artifact_path="model",
        #     signature=signature,
        #     input_example=sample_input_np,
        #     registered_model_name=registered_model_name,
        # )

        # new_version = model_info.registered_model_version

        # client.set_registered_model_alias(
        #     registered_model_name,
        #     "prd",
        #     new_version
        # )

        mlflow.set_tags({
            "framework": "PyTorch",
            "task": "multilabel_classification"
        })

        print("Run ID:", mlflow.active_run().info.run_id)

        print("Registered model version:", new_version)

        print("Alias 'prd' ->", new_version)

    return train_losses, val_losses, val_rocs

In [22]:
class BaselineToxNet(nn.Module):
  def __init__(self, input_size, output_size):
        super(BaselineToxNet, self).__init__()

        self.fc1 = nn.Linear(input_size, 128)
        self.relu1 = nn.ReLU()

        self.dropout = nn.Dropout(0.2)

        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()

        self.fc3 = nn.Linear(64, output_size)

  def forward(self, x):
    out = self.fc1(x)
    out = self.relu1(out)
    out = self.dropout(out)
    out = self.fc2(out)
    out = self.relu2(out)
    out = self.dropout(out)
    out = self.fc3(out)
    return out

In [23]:
criterion= nn.BCEWithLogitsLoss()

In [24]:
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

X_train_torch = torch.tensor(X_train, dtype=torch.float32)
y_train_torch = torch.tensor(y_train[target_cols].values, dtype = torch.float32)

X_val_torch = torch.tensor(X_val, dtype=torch.float32)
y_val_torch = torch.tensor(y_val.values, dtype=torch.float32)


train_dataset = TensorDataset(X_train_torch, y_train_torch)
val_dataset = TensorDataset(X_val_torch, y_val_torch)


train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)

In [ ]:
experiment_name = "toxicity-pytorch"
artifact_location = "mlflow-artifacts:/"

client = MlflowClient()

exp = mlflow.get_experiment_by_name(experiment_name)
if exp is None:
    exp_id = client.create_experiment(
        name=experiment_name,
        artifact_location=artifact_location
    )
else:
    exp_id = exp.experiment_id

mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='mlflow-artifacts:/', creation_time=1779893556252, experiment_id='6', last_update_time=1779893556252, lifecycle_stage='active', name='torch', tags={}>

In [34]:
model = BaselineToxNet(X_train.shape[1], y_train.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

registered_model_name = "nn_base"

epoch = 20
batch_size = 512


In [35]:
train_losses, val_losses, val_rocs = train_and_validate(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=20,
    device=device,
    registered_model_name="registered_model_name",
    experiment_name=experiment_name)

Epoch 1 | Train Loss=0.2624 | Val Loss=0.2326 | ROC-AUC=0.7058
Epoch 2 | Train Loss=0.2352 | Val Loss=0.2236 | ROC-AUC=0.7469
Epoch 3 | Train Loss=0.2283 | Val Loss=0.2193 | ROC-AUC=0.7524
Epoch 4 | Train Loss=0.2250 | Val Loss=0.2164 | ROC-AUC=0.7690
Epoch 5 | Train Loss=0.2215 | Val Loss=0.2139 | ROC-AUC=0.7702
Epoch 6 | Train Loss=0.2187 | Val Loss=0.2110 | ROC-AUC=0.7841
Epoch 7 | Train Loss=0.2170 | Val Loss=0.2095 | ROC-AUC=0.7838
Epoch 8 | Train Loss=0.2151 | Val Loss=0.2082 | ROC-AUC=0.7860
Epoch 9 | Train Loss=0.2130 | Val Loss=0.2076 | ROC-AUC=0.7913
Epoch 10 | Train Loss=0.2118 | Val Loss=0.2068 | ROC-AUC=0.7950
Epoch 11 | Train Loss=0.2105 | Val Loss=0.2055 | ROC-AUC=0.7961
Epoch 12 | Train Loss=0.2088 | Val Loss=0.2059 | ROC-AUC=0.7946
Epoch 13 | Train Loss=0.2075 | Val Loss=0.2037 | ROC-AUC=0.7980
Epoch 14 | Train Loss=0.2067 | Val Loss=0.2043 | ROC-AUC=0.8040
Epoch 15 | Train Loss=0.2063 | Val Loss=0.2026 | ROC-AUC=0.8003
Epoch 16 | Train Loss=0.2053 | Val Loss=0.2015 | 

2026/05/27 18:15:37 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'registered_model_name'.
2026/05/27 18:15:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: registered_model_name, version 1
Created version '1' of model 'registered_model_name'.


Run ID: 1986e11ccf664f6a9b7f3d139c5e801f
Registered model version: 1
Alias 'prd' -> 1
🏃 View run registered_model_name at: http://localhost:5050/#/experiments/6/runs/1986e11ccf664f6a9b7f3d139c5e801f
🧪 View experiment at: http://localhost:5050/#/experiments/6


In [36]:
loaded_model = mlflow.pytorch.load_model(f"models:/{registered_model_name}@prd")
loaded_model.eval()

with torch.no_grad():
    sample_logits = loaded_model(X_val_torch)
    sample_pred = torch.argmax(sample_logits, dim=1).numpy()

print("Sample classes:", sample_pred)

/Users/kuzmndmtry/ml/hse/yp/27_toxicity_prediction/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

Sample classes: [11 11 11 ...  9 11 11]


In [29]:
print("Final ROC-AUC:", val_rocs[-1])

Final ROC-AUC: 0.819751958731977


In [30]:
class BatchmormToxNet(nn.Module):
  def __init__(self, input_size, output_size):
        super(BatchmormToxNet, self).__init__()

        self.fc1 = nn.Linear(input_size, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.relu1 = nn.ReLU()



        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.relu2 = nn.ReLU()

        self.dropout = nn.Dropout(0.2)

        self.fc3 = nn.Linear(64, output_size)

  def forward(self, x):
    out = self.fc1(x)
    out = self.bn1(out)
    out = self.relu1(out)
    out = self.dropout(out)

    out = self.fc2(out)
    out = self.bn2(out)
    out = self.relu2(out)
    out = self.dropout(out)

    out = self.fc3(out)
    return out

In [ ]:
bn_model = BatchmormToxNet(X_train.shape[1], y_train.shape[1]).to(device)
optimizer = torch.optim.Adam(bn_model.parameters(), lr = 0.001)

registered_model_name = "nn_Batchmorm"

epoch = 20
batch_size = 512


In [ ]:
train_losses, val_losses, val_rocs = train_and_validate(
    model=bn_model,
    optimizer=optimizer,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=20,
    device=device,
    registered_model_name=registered_model_name,
    experiment_name="toxicity-pytorch")

Epoch 1 | Train Loss=0.2668 | Val Loss=0.2253 | ROC-AUC=0.7367
Epoch 2 | Train Loss=0.2302 | Val Loss=0.2184 | ROC-AUC=0.7625
Epoch 3 | Train Loss=0.2248 | Val Loss=0.2149 | ROC-AUC=0.7840
Epoch 4 | Train Loss=0.2213 | Val Loss=0.2119 | ROC-AUC=0.7838
Epoch 5 | Train Loss=0.2184 | Val Loss=0.2103 | ROC-AUC=0.7958
Epoch 6 | Train Loss=0.2163 | Val Loss=0.2088 | ROC-AUC=0.7980
Epoch 7 | Train Loss=0.2143 | Val Loss=0.2063 | ROC-AUC=0.8006
Epoch 8 | Train Loss=0.2126 | Val Loss=0.2054 | ROC-AUC=0.8024
Epoch 9 | Train Loss=0.2117 | Val Loss=0.2051 | ROC-AUC=0.8083
Epoch 10 | Train Loss=0.2099 | Val Loss=0.2038 | ROC-AUC=0.8091
Epoch 11 | Train Loss=0.2085 | Val Loss=0.2041 | ROC-AUC=0.8079
Epoch 12 | Train Loss=0.2073 | Val Loss=0.2023 | ROC-AUC=0.8114
Epoch 13 | Train Loss=0.2062 | Val Loss=0.2023 | ROC-AUC=0.8061
Epoch 14 | Train Loss=0.2061 | Val Loss=0.2008 | ROC-AUC=0.8137
Epoch 15 | Train Loss=0.2054 | Val Loss=0.2004 | ROC-AUC=0.8105
Epoch 16 | Train Loss=0.2042 | Val Loss=0.2008 | 

2026/05/27 17:34:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'nn_Batchmorm'.
2026/05/27 17:35:00 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: nn_Batchmorm, version 1


Run ID: a1910f97a0be4c64b225d81fbdb88acc
Registered model version: 1
Alias 'prd' -> 1
🏃 View run nn_Batchmorm at: http://localhost:5050/#/experiments/5/runs/a1910f97a0be4c64b225d81fbdb88acc
🧪 View experiment at: http://localhost:5050/#/experiments/5


Created version '1' of model 'nn_Batchmorm'.


In [40]:
def weighted_bce_with_logits(pos_weight):
    return nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [41]:
pos_weight = (y_train == 0).sum(axis=0) / (y_train == 1).sum(axis=0)
pos_weight = torch.tensor(pos_weight, dtype=torch.float32).to(device)

/var/folders/zg/5g1zcx7n41d85mldns1g6v0h0000gn/T/ipykernel_38788/2559209719.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pos_weight = torch.tensor(pos_weight, dtype=torch.float32).to(device)


In [38]:
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )

        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.block(x))

In [39]:
class StrongToxNet(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()

        self.input = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU()
        )

        self.res1 = ResidualBlock(256)
        self.res2 = ResidualBlock(256)

        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, output_size)
        )

    def forward(self, x):
        out = self.input(x)
        out = self.res1(out)
        out = self.res2(out)
        out = self.head(out)
        return out

In [47]:
rb_model = StrongToxNet(X_train.shape[1], y_train.shape[1]).to(device)
optimizer = torch.optim.Adam(rb_model.parameters(), lr = 0.001)
registered_model_name = "nn_Strong"
epoch = 20
batch_size = 512

In [48]:
train_losses, val_losses, val_rocs = train_and_validate(
    model=rb_model,
    optimizer=optimizer,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=20,
    device=device,
    registered_model_name=registered_model_name,
    experiment_name="toxicity-pytorch")

Epoch 1 | Train Loss=0.2400 | Val Loss=0.2180 | ROC-AUC=0.7816
Epoch 2 | Train Loss=0.2190 | Val Loss=0.2099 | ROC-AUC=0.7977
Epoch 3 | Train Loss=0.2106 | Val Loss=0.2042 | ROC-AUC=0.8123
Epoch 4 | Train Loss=0.2040 | Val Loss=0.2011 | ROC-AUC=0.8146
Epoch 5 | Train Loss=0.1988 | Val Loss=0.1985 | ROC-AUC=0.8162
Epoch 6 | Train Loss=0.1943 | Val Loss=0.1980 | ROC-AUC=0.8175
Epoch 7 | Train Loss=0.1895 | Val Loss=0.1958 | ROC-AUC=0.8179
Epoch 8 | Train Loss=0.1859 | Val Loss=0.1948 | ROC-AUC=0.8206
Epoch 9 | Train Loss=0.1812 | Val Loss=0.1942 | ROC-AUC=0.8126
Epoch 10 | Train Loss=0.1776 | Val Loss=0.1939 | ROC-AUC=0.8214
Epoch 11 | Train Loss=0.1743 | Val Loss=0.1964 | ROC-AUC=0.8178
Epoch 12 | Train Loss=0.1709 | Val Loss=0.1915 | ROC-AUC=0.8234
Epoch 13 | Train Loss=0.1671 | Val Loss=0.1946 | ROC-AUC=0.8264
Epoch 14 | Train Loss=0.1643 | Val Loss=0.1960 | ROC-AUC=0.8151
Epoch 15 | Train Loss=0.1606 | Val Loss=0.1964 | ROC-AUC=0.8179
Epoch 16 | Train Loss=0.1577 | Val Loss=0.1954 | 

2026/05/27 18:25:31 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'nn_Strong' already exists. Creating a new version of this model...
2026/05/27 18:25:32 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: nn_Strong, version 3


Run ID: 7130c25e3d21493f928f346e54e2382c
Registered model version: 3
Alias 'prd' -> 3
🏃 View run nn_Strong at: http://localhost:5050/#/experiments/5/runs/7130c25e3d21493f928f346e54e2382c
🧪 View experiment at: http://localhost:5050/#/experiments/5


Created version '3' of model 'nn_Strong'.


In [46]:
loaded_model = mlflow.pytorch.load_model(f"models:/{registered_model_name}@prd")
loaded_model.eval()

with torch.no_grad():
    sample_logits = loaded_model(X_val_torch)
    sample_pred = torch.argmax(sample_logits, dim=1).numpy()

print("Sample classes:", sample_pred)

Sample classes: [11 11 11 ...  9  9 11]


Дальше сырые ансамбли

In [113]:
models = {}
feature_names_dict = {}

for file in os.listdir('/content/drive/MyDrive/tox (1)/XGBoost_50/'):
    if file.endswith('.joblib'):
        cat = file.replace('xgb_', '').replace('.joblib', '')
        models[cat] = joblib.load(f'/content/drive/MyDrive/tox (1)/XGBoost_50/{file}')

        feature_names = models[cat].get_booster().feature_names
        feature_names_dict[cat] = feature_names


In [118]:
feature_idx = list(map(int, feature_names_dict['XGBoost_50_cardiotoxicity']))

In [119]:
X_val_top50 = X_val[:, feature_idx]

In [121]:
preds = {}

for cat, model in models.items():
    preds[cat] = model.predict_proba(X_val_top50)[:, 1]

In [130]:
xgb_preds = np.column_stack([
    preds[f"XGBoost_50_{cat}"] for cat in target_cols
])

In [139]:
X_val = X_val.values if hasattr(X_val, "values") else X_val
y_val = y_val[target_cols].values if hasattr(y_val, "values") else y_val

In [140]:
target_cols = list(target_cols)  # фиксируем порядок
n_tasks = len(target_cols)

In [141]:
target_cols = list(target_cols)  # фиксируем порядок
n_tasks = len(target_cols)

In [143]:
rb_model.eval()

all_preds = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)

        outputs = rb_model(X_batch)        # logits
        probs = torch.sigmoid(outputs)   # вероятности

        all_preds.append(probs.cpu().numpy())

In [144]:
nn_preds = np.vstack(all_preds)

In [145]:
import numpy as np
from sklearn.metrics import roc_auc_score

weights = np.linspace(0, 1, 21)

best_w = {}
best_score = {}

for i, cat in enumerate(target_cols):

    y = y_val[:, i]
    mask = ~np.isnan(y)

    best_local_w = 0
    best_local_score = -1

    for w in weights:

        preds_ens = (
            w * xgb_preds[:, i] +
            (1 - w) * nn_preds[:, i]
        )

        score = roc_auc_score(y[mask], preds_ens[mask])

        if score > best_local_score:
            best_local_score = score
            best_local_w = w

    best_w[cat] = best_local_w
    best_score[cat] = best_local_score

    print(f"{cat}: w={best_local_w:.2f}, ROC-AUC={best_local_score:.4f}")

acute_toxicity: w=1.00, ROC-AUC=0.9451
carcinogenicity: w=1.00, ROC-AUC=0.9808
cardiotoxicity: w=1.00, ROC-AUC=0.9858
dermal_toxicity: w=0.95, ROC-AUC=0.9824
genotoxicity: w=1.00, ROC-AUC=0.9917
hepatotoxicity: w=0.85, ROC-AUC=0.9797
ocular_toxicity: w=1.00, ROC-AUC=0.9769
oxidative_stress: w=1.00, ROC-AUC=0.9420
respiratory_toxicity: w=1.00, ROC-AUC=0.9731
neuro_sensory_toxicity: w=0.85, ROC-AUC=0.9897
immuno_hematotoxicity: w=1.00, ROC-AUC=0.9471
reprod_dev_toxicity: w=0.35, ROC-AUC=0.9663
endocrine_metabolic_tox: w=1.00, ROC-AUC=0.9244


In [147]:
torch.save(rb_model.state_dict(), "rb_model.pt")

In [152]:
load_model = StrongToxNet(X_train.shape[1], y_train.shape[1]).to(device)

In [153]:
load_model.load_state_dict(torch.load("rb_model.pt", map_location=device))
load_model.eval()

StrongToxNet(
  (input): Sequential(
    (0): Linear(in_features=78, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (res1): ResidualBlock(
    (block): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=256, out_features=256, bias=True)
      (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (relu): ReLU()
  )
  (res2): ResidualBlock(
    (block): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=256, out_features=256, bias=True)
      (5): Ba